# PCB test-point placement, static canonical board

Single-instance optimization on the real TE AutoLayout Example01 board (135x90mm, 20 traces, `AutoLayout_Example01.xlsx`): 0 planar failures, then min max trace, then min total, and the placement must serpentine-equalize (matched 20/20).

Where things stand:
- Classical baseline (`smart_placement`): 20/20 routed, max 95mm.
- The 60k cold-start run in `pcb-router-logs-v4/canonical` clones that solution. On a static board the policy adds nothing over search; the run's value is its world model.
- Blind router-in-the-loop search reached ~86mm ungated in a 200s probe.
- Current experiment: `scripts/wm_guided_search.py` uses the world model as a router surrogate to guide search.

Setup: A100 runtime (Colab Pro), run the setup cells top to bottom once, then run whichever phase you need. Training is already done; the train cell resumes, it never starts over.

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), "No GPU: set Runtime > Change runtime type > A100 GPU"
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
LOGROOT = "/content/drive/MyDrive/pcb-router-logs-v4"
print(LOGROOT)

In [ ]:
import pathlib
if not pathlib.Path("/content/pcb-router").exists():
    !git clone -q https://github.com/pauljiang03/pcb-router /content/pcb-router
%cd /content/pcb-router
!git pull --ff-only

In [ ]:
%pip -q install gymnasium "ruamel.yaml" openpyxl
!python -m pytest tests/test_coldstart.py tests/test_wm_search.py -q

In [ ]:
NUM_TRACES = 20
CONFIG = "colab_a100"  # T4 fallback: "colab"
BOARDS = "canonical"   # the exact xlsx board, static every episode
STEPS = 60000
DEMOS = 25
ENV_FLAGS = "--envs 4 --parallel"
EVAL_FLAGS = "--board canonical" if "canonical" in BOARDS else ""
RUN_DIR = f"{LOGROOT}/canonical"  # completed 60k run; point at an empty dir to retrain
CKPT = f"{RUN_DIR}/latest.pt"
print(NUM_TRACES, "traces |", CONFIG, "|", BOARDS, "|", RUN_DIR)

In [ ]:
%load_ext tensorboard
import tensorboard.notebook as tbnb
tbnb.start("--logdir " + LOGROOT)

## Train (done; resumes, never restarts)

The 60k cold-start run (demos, anchored BC, potential shaping, single-layer reward) is complete; re-running this cell extends it by at most one 5k chunk. For a fresh retrain, point RUN_DIR at an empty dir first. A fresh run generates demos (~2 min), then trains 60k steps (2-4 h, checkpoints every 5k, interrupt-safe).

In [ ]:
!python train.py --configs defaults {CONFIG} --logdir "{RUN_DIR}" \
    --num_traces {NUM_TRACES} --device cuda:0 {ENV_FLAGS} \
    --boards {BOARDS} --steps {STEPS} --demos {DEMOS}

In [ ]:
# Scoreboard vs classical baselines on the same board (--fast for a quick pass).
!python eval.py --checkpoint "{CKPT}" --configs defaults {CONFIG} \
    --episodes 3 --num_traces {NUM_TRACES} --device cuda:0 --no-plot {EVAL_FLAGS}

In [ ]:
# Render routed boards to PNGs (~2-3 min) and show Smart vs Dreamer.
!python eval.py --checkpoint "{CKPT}" --configs defaults {CONFIG} \
    --episodes 1 --num_traces {NUM_TRACES} --device cuda:0 {EVAL_FLAGS}

import pathlib, shutil
from IPython.display import Image, display
figs = pathlib.Path(RUN_DIR) / "figs"
figs.mkdir(exist_ok=True)
for p in sorted(pathlib.Path("eval_results").glob("*_1.png")):
    shutil.copy(p, figs / p.name)
for name in ("smart_1.png", "dreamer_1.png"):
    p = pathlib.Path("eval_results") / name
    if p.exists():
        display(Image(str(p)))

## Search then distill (expert iteration)

Router-in-the-loop local search beats the policy on a static board (a 200s probe took 95mm/1332mm to 86mm/1073mm, ungated). Distill regenerates the demos from the optimized placement and resumes training anchored to it, so the policy reproduces the search result one-shot.

In [ ]:
# Search: verifies routing and equalization (matched=20/20) at the end.
!python scripts/optimize_placement.py --board canonical --minutes 15 \
    --out "{RUN_DIR}/best_placement.json"

In [ ]:
# Distill: full-strength anchor, the policy clones the search result one-shot.
!rm -f "{RUN_DIR}"/demo_eps/*.npz
!python train.py --configs defaults {CONFIG} --logdir "{RUN_DIR}" \
    --num_traces {NUM_TRACES} --device cuda:0 {ENV_FLAGS} \
    --boards {BOARDS} --steps 90000 --demos {DEMOS} \
    --demo_placement "{RUN_DIR}/best_placement.json" --bc_scale 10

## World-model-guided search (current experiment)

Observations are pure geometry (routing only enters the terminal reward), so any placement's full episode can be built without routing and scored by the reward head in one batched forward. Search scores mutations with that surrogate and spends real router calls only on the top-k. Reports surrogate fidelity (Spearman rho vs true returns, plus rho vs -max on the routable subset) and guided vs blind at equal router calls; the blind arm continues to 3x C to measure headroom. The output JSON is --demo_placement compatible.

In [ ]:
# The blind phase can take up to ~3x the guided minutes.
!python scripts/wm_guided_search.py --checkpoint "{CKPT}" \
    --configs defaults colab_a100 --minutes 10 --fidelity 32 \
    --device cuda:0 --out "{RUN_DIR}/wm_search.json"

## Reading the results

wm_search.json: fidelity rho >= 0.6 means the surrogate genuinely ranks placements; 0.3-0.6 is partial guidance; below 0.3 the reward head does not rank off-policy mutations, and the honest extension is fine-tuning it on search trial logs, not shipping a weak result. Also check spearman_neg_max_zero_fail (rho against the actual search objective). The speedup claim holds if calls_to_match_guided is at least 3x guided's router_calls; 0 means guided never beat the shared seed's max.

Training curves (only when retraining): log_routable should sit at +10 (each planar failure costs 5); bc_loss falls toward ~1 and never vanishes (permanent 10% anchor); imag_reward_mean above ~+2 means the reward head is hallucinating; eval_return should reach the demo anchor printed during demo generation.

Everything is backed up in Drive; stopping the runtime loses nothing.

In [ ]:
# Optional local copy; Drive already has everything.
import pathlib, shutil
out = pathlib.Path("/content/results")
shutil.rmtree(out, ignore_errors=True)
out.mkdir(parents=True)
d = pathlib.Path(RUN_DIR)
for f in list(d.glob("events*")) + [d / "latest.pt", d / "best_placement.json",
                                    d / "wm_search.json"]:
    if f.exists():
        shutil.copy(f, out / f.name)
shutil.make_archive("/content/pcb_router_results", "zip", out)
from google.colab import files
files.download("/content/pcb_router_results.zip")